### Import

In [2]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from matplotlib import pyplot as plt 
from hcuppy.elixhauser import ElixhauserEngine

pd.set_option('display.max_columns', 500)

### General parameters

In [3]:
eicu  = "PATH TO DATA/eICU/"
output  = "../Data/csvExtract/"

In [4]:
Embed_dim = 20 # 10, 20, 50

### Read icd diagnosis

In [5]:
diagnose = pd.read_csv(eicu + "diagnosis.csv")
diagnose = diagnose[['patientunitstayid', 'diagnosisoffset', 'icd9code']]

In [6]:
diagnose = diagnose[diagnose.icd9code.notnull()]
diagnose = diagnose.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True])
diagnose['icdCount'] = diagnose.icd9code.apply(lambda x: len(x.split(',')))
diagnose[['a', 'b', 'c', 'd', 'e', 'f', 'g']] = diagnose['icd9code'].str.split(pat=",", expand=True, n=-1)
diagnose.drop(columns=['icd9code', 'icdCount'], inplace=True)

In [7]:
diagnose['a'] = diagnose['a'].str.replace(' ', '')
diagnose['b'] = diagnose['b'].str.replace(' ', '')
diagnose['c'] = diagnose['c'].str.replace(' ', '')
diagnose['d'] = diagnose['d'].str.replace(' ', '')
diagnose['e'] = diagnose['e'].str.replace(' ', '')
diagnose['f'] = diagnose['f'].str.replace(' ', '')
diagnose['g'] = diagnose['g'].str.replace(' ', '')

diagnose = pd.melt(diagnose, id_vars=['patientunitstayid', 'diagnosisoffset'], 
                   value_vars=['a', 'b', 'c', 'd', 'e', 'f', 'g'], var_name='myVarname', value_name='icd9_code')

In [8]:
diagnose.drop(columns=['myVarname'], inplace=True)
diagnose = diagnose[diagnose.icd9_code.notnull()]
diagnose['icd10_code'] = diagnose.icd9_code
diagnose.loc[~(diagnose['icd10_code'].str.match('^[A-Z].*') == True), 'icd10_code'] = np.nan
diagnose.loc[diagnose.icd10_code.notnull(), 'icd9_code'] = np.nan
diagnose = diagnose.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True]).reset_index(drop=True)

### Seperating ICD-9 and ICD-10

In [9]:
diagnose_icd9  = diagnose[['patientunitstayid', 'diagnosisoffset', 'icd9_code']]
diagnose_icd10 = diagnose[['patientunitstayid', 'diagnosisoffset', 'icd10_code']]

diagnose_icd9  = diagnose_icd9[diagnose_icd9.icd9_code.notnull()]
diagnose_icd10 = diagnose_icd10[diagnose_icd10.icd10_code.notnull()]

diag_icd10_rep = diagnose_icd10.copy()

diagnose_icd9 = diagnose_icd9.groupby(['patientunitstayid', 'diagnosisoffset'])['icd9_code'].agg(['unique'])
diagnose_icd10 = diagnose_icd10.groupby(['patientunitstayid', 'diagnosisoffset'])['icd10_code'].agg(['unique'])

diagnose_icd9 = diagnose_icd9.reset_index()
diagnose_icd10 = diagnose_icd10.reset_index()

diagnose_icd9.rename(columns={"unique": "ICD-9"}, inplace=True)
diagnose_icd10.rename(columns={"unique": "ICD-10"}, inplace=True)

diagnose_icd9 = diagnose_icd9.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True]).reset_index(drop=True)
diagnose_icd10 = diagnose_icd10.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True]).reset_index(drop=True)

### ICD-10 embeddings representation

In [10]:
diag_icd10_rep['ICD_10'] = diag_icd10_rep.icd10_code

In [11]:
diag_icd10_rep.ICD_10 = diag_icd10_rep.ICD_10.astype(str)
diag_icd10_rep.ICD_10 = diag_icd10_rep.ICD_10.str.replace('.', '')
diag_icd10_rep['ICD_10'] = diag_icd10_rep['ICD_10'].astype(str).str[0:5]

<ipython-input>:2: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  diag_icd10_rep.ICD_10 = diag_icd10_rep.ICD_10.str.replace('.', '')


In [12]:
meta_data = pd.read_csv('../Data/Embeddings_ICD10/Meta.tsv', sep='\t', names=['col'])
meta_data[['ICD_10','name_of_code']] = meta_data["col"].str.split(" ", 1, expand=True)
meta_data.drop(columns=['col', 'name_of_code'], inplace=True)

<ipython-input>:2: FutureWarning: In a future version of pandas all arguments of StringMethods.split except for the argument 'pat' will be keyword-only.
  meta_data[['ICD_10','name_of_code']] = meta_data["col"].str.split(" ", 1, expand=True)


In [13]:
if Embed_dim == 10:
    vec_embd = pd.read_csv('../Data/Embeddings_ICD10/10d-vecs.tsv', sep='\t', header=None)
elif Embed_dim == 20:
    vec_embd = pd.read_csv('../Data/Embeddings_ICD10/20d-vecs.tsv', sep='\t', header=None)    
elif Embed_dim == 50:
    vec_embd = pd.read_csv('../Data/Embeddings_ICD10/50d-vecs.tsv', sep='\t', header=None)    
    
embeddings = pd.concat([meta_data, vec_embd], axis=1)
embeddings = embeddings.round(3)

In [14]:
columns = [i for i in range(Embed_dim)]
mean_embedding = pd.DataFrame(embeddings[columns].mean()).T.reset_index(drop=True)

In [15]:
ICD_4d = embeddings.copy()
ICD_4d['ICD_10'] = ICD_4d['ICD_10'].astype(str).str[0:4]
ICD_4d = ICD_4d.groupby('ICD_10').mean().reset_index()

ICD_3d = embeddings.copy()
ICD_3d['ICD_10'] = ICD_3d['ICD_10'].astype(str).str[0:3]
ICD_3d = ICD_3d.groupby('ICD_10').mean().reset_index()

ICD_2d = embeddings.copy()
ICD_2d['ICD_10'] = ICD_2d['ICD_10'].astype(str).str[0:2]
ICD_2d = ICD_2d.groupby('ICD_10').mean().reset_index()

### Merge ICD codes with embeddings

In [16]:
diag_icd10_rep = pd.merge(diag_icd10_rep, embeddings, on='ICD_10', how='left')

In [17]:
df1 = diag_icd10_rep[diag_icd10_rep[0].notnull()]
df2 = diag_icd10_rep[diag_icd10_rep[0].isnull()]
df2 = df2.drop(columns, axis=1)
df2['ICD_10'] = df2['ICD_10'].astype(str).str[0:4]
df2 = pd.merge(df2, ICD_4d, on='ICD_10', how='left')
frames = [df1, df2]
diag_icd10_rep = pd.concat(frames).reset_index(drop=True)

In [18]:
df1 = diag_icd10_rep[diag_icd10_rep[0].notnull()]
df2 = diag_icd10_rep[diag_icd10_rep[0].isnull()]
df2 = df2.drop(columns, axis=1)
df2['ICD_10'] = df2['ICD_10'].astype(str).str[0:3]
df2 = pd.merge(df2, ICD_3d, on='ICD_10', how='left')
frames = [df1, df2]
diag_icd10_rep = pd.concat(frames).reset_index(drop=True)

In [19]:
df1 = diag_icd10_rep[diag_icd10_rep[0].notnull()]
df2 = diag_icd10_rep[diag_icd10_rep[0].isnull()]
df2 = df2.drop(columns, axis=1)
df2['ICD_10'] = df2['ICD_10'].astype(str).str[0:2]
df2 = pd.merge(df2, ICD_2d, on='ICD_10', how='left')
frames = [df1, df2]
diag_icd10_rep = pd.concat(frames).reset_index(drop=True)

In [20]:
reimburcement = ['E980.2', 'E980.4', 'E980.1', 'E980.5', 'E980.3', 'E982.1', 'E980.0', 'E924.1', 'E932.3', 
                 'E942.9', 'E980.8', 'E933.1', 'E947.8', 'E980.7', 'E930.8', 'E935.3', 'E934.2', 'E930.1', 
                 'E945.2', 'E932.0']

diag_icd10_rep = diag_icd10_rep[~diag_icd10_rep['icd10_code'].isin(reimburcement)]

In [21]:
diag_icd10_rep.loc[diag_icd10_rep[0].isnull(), columns] = mean_embedding[columns].values
diag_icd10_rep[columns] = diag_icd10_rep[columns].round(3)

In [22]:
diag_icd10_rep.drop(columns=['icd10_code', 'ICD_10'], inplace=True)
diag_icd10_rep = diag_icd10_rep.groupby(['patientunitstayid', 'diagnosisoffset']).mean().reset_index().round(3)

In [23]:
diag_icd10_rep['ICD-10_Embedding']= diag_icd10_rep[columns].values.tolist()
diag_icd10_rep.drop(columns=columns, inplace=True)

In [24]:
diag_icd10_rep.head(3)

### Calculating Elixhauser Scores (Readmission, Mortality, Comorbidity)

In [25]:
def elixhauser(icd10_list):
    
    ee = ElixhauserEngine()
    out = ee.get_elixhauser(list(icd10_list))
    readm = out["rdmsn_scr"]
    mortal= out["mrtlt_scr"]
    comor = out["cmrbdt_lst"]
    
    return readm, mortal, comor

In [26]:
diagnose_icd10['readmission_scr'], diagnose_icd10['mortality_risks_scr'], diagnose_icd10['comorbidity_lst'] = np.nan, np.nan, np.nan
diagnose_icd10['readmission_scr'], diagnose_icd10['mortality_risks_scr'], diagnose_icd10['comorbidity_lst']  = zip(*diagnose_icd10['ICD-10'].map(elixhauser))

In [27]:
diagnose_icd10 = diagnose_icd10.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True]).reset_index(drop=True)

In [28]:
Elixhauser = diagnose_icd10[['patientunitstayid', 'diagnosisoffset', 'comorbidity_lst', 
                             'readmission_scr', 'mortality_risks_scr']]

diagnose_icd10 = diagnose_icd10[['patientunitstayid', 'diagnosisoffset', 'ICD-10']]

In [29]:
diagnose_icd10 = pd.merge(diagnose_icd10, diag_icd10_rep, on=['patientunitstayid', 'diagnosisoffset'], how='left')

### Final Data

In [30]:
Elixhauser.head(3)

In [31]:
diagnose_icd9.head(3)

In [32]:
diagnose_icd10.head(3)

### Save Data

In [33]:
Elixhauser.to_csv(output + 'elixhauser.csv',  index=False)
diagnose_icd9.to_csv(output + 'diagnose_icd9.csv',  index=False)
diagnose_icd10.to_csv(output + 'diagnose_icd10.csv', index=False)